# Ablation Analysis

Analyses the four forecasting decoder variants (full model, w/o query self-attn, w/o query cross-attn, plain decoder) across 3 seeds each.

Reads pre-computed `heldout_metrics.json` per variant/seed and the pre-aggregated `ablation_summary_mean_std.csv`.
Joins with benchmarking results from notebook 05 where available.

In [ ]:
from pathlib import Path
import sys
import json
import warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.utils.config import load_yaml_config

# Optional plotting imports
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False
    print('[WARN] matplotlib not available; figures will be skipped')

print('ROOT:', ROOT)

In [ ]:
# ===== Parameters =====
SMOKE_MODE = True            # False -> fuller reporting / extra plots
SEED = 42

# Input paths
ABLATION_DIR = ROOT / 'checkpoints' / 'forecasting_ablation'
BENCHMARK_DIR = ROOT / 'artifacts' / 'notebooks' / 'benchmarking'

# Output paths
OUTDIR = ROOT / 'artifacts' / 'notebooks' / 'ablation'
FIGDIR = OUTDIR / 'figures'
OUTDIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

# Ablation config (for variant metadata)
ABL_CFG_PATH = ROOT / 'configs' / 'forecasting_ablation.yaml'
ABL_CFG = load_yaml_config(ABL_CFG_PATH) if ABL_CFG_PATH.exists() else {}

# Variant definitions (hardcoded fallback if config missing)
VARIANT_NAMES = ['full_model', 'wo_query_self_attention', 'wo_query_cross_attention', 'plain_decoder']
VARIANT_LABELS = {
    'full_model': 'Full model',
    'wo_query_self_attention': 'w/o query self-attn',
    'wo_query_cross_attention': 'w/o query cross-attn',
    'plain_decoder': 'Plain decoder',
}

# Priority metrics and their direction (+1 = higher is better, -1 = lower is better)
METRIC_DIRECTION = {
    'heldout_mse': -1,
    'global_corr_nonzero': +1,
    'meta_cell_corr_by_time_bin': +1,
    'sparsity_gap': -1,
    'asw_cell_type_separation': +1,
    'ari': +1,
    'nmi': +1,
}

print('SMOKE_MODE:', SMOKE_MODE)
print('ABLATION_DIR:', ABLATION_DIR)
print('BENCHMARK_DIR:', BENCHMARK_DIR)
print('OUTDIR:', OUTDIR)

In [ ]:
# ===== Scan ablation directory and read all heldout_metrics.json =====
records = []
missing_files = []
found_files = []

for variant_name in VARIANT_NAMES:
    variant_dir = ABLATION_DIR / variant_name
    if not variant_dir.is_dir():
        missing_files.append(f'{variant_name}/ (directory missing)')
        continue

    for seed_dir in sorted(variant_dir.iterdir()):
        if not seed_dir.is_dir():
            continue
        metrics_file = seed_dir / 'heldout_metrics.json'
        if metrics_file.exists():
            try:
                with open(metrics_file) as f:
                    rec = json.load(f)
                rec['_source_file'] = str(metrics_file.relative_to(ROOT))
                records.append(rec)
                found_files.append(str(metrics_file.relative_to(ROOT)))
            except Exception as e:
                missing_files.append(f'{metrics_file.relative_to(ROOT)}: {e}')
        else:
            missing_files.append(str(metrics_file.relative_to(ROOT)))

per_seed_df = pd.DataFrame(records)
print(f'\nFound {len(per_seed_df)} heldout_metrics.json files across {per_seed_df["variant"].nunique() if "variant" in per_seed_df.columns else 0} variants')

if missing_files:
    print(f'\n[WARN] {len(missing_files)} missing/unreadable files:')
    for mf in missing_files[:10]:
        print(f'  - {mf}')
    if len(missing_files) > 10:
        print(f'  ... and {len(missing_files) - 10} more')

if not per_seed_df.empty:
    print('\nColumns:', list(per_seed_df.columns))
    display(per_seed_df[['variant', 'label', 'seed', 'heldout_mse', 'global_corr_nonzero']].head(12))

In [ ]:
# ===== Also try reading pre-aggregated CSVs =====

# 1) ablation_summary_mean_std.csv
summary_csv = ABLATION_DIR / 'ablation_summary_mean_std.csv'
if summary_csv.exists():
    try:
        # This CSV has a two-line header: metric names on row 0, mean/std on row 1
        raw = pd.read_csv(summary_csv, header=None)
        # Build clean column names
        metrics_row = raw.iloc[0].fillna(method='ffill').tolist()
        stat_row = raw.iloc[1].fillna('value').tolist()
        col_names = [f'{m}_{s}' if s != 'value' and pd.notna(s) and s != '' else str(m)
                     for m, s in zip(metrics_row, stat_row)]
        col_names[0] = 'variant'
        col_names[1] = 'label'
        summary_df = raw.iloc[2:].copy()
        summary_df.columns = col_names
        summary_df = summary_df.reset_index(drop=True)
        for col in summary_df.columns[2:]:
            summary_df[col] = pd.to_numeric(summary_df[col], errors='coerce')
        print(f'Loaded ablation_summary_mean_std.csv: {len(summary_df)} variants')
        display(summary_df)
    except Exception as e:
        print(f'[WARN] Failed to parse ablation_summary_mean_std.csv: {e}')
        summary_df = pd.DataFrame()
else:
    print('[WARN] ablation_summary_mean_std.csv not found')
    summary_df = pd.DataFrame()

# 2) ablation_results.csv (per-seed, same as heldout_metrics.json but in CSV)
results_csv = ABLATION_DIR / 'ablation_results.csv'
if results_csv.exists():
    try:
        results_df = pd.read_csv(results_csv)
        results_df = results_df.dropna(how='all')
        print(f'\nLoaded ablation_results.csv: {len(results_df)} rows')
    except Exception as e:
        print(f'[WARN] Failed to parse ablation_results.csv: {e}')
        results_df = pd.DataFrame()
else:
    results_df = pd.DataFrame()

In [ ]:
# ===== Build ablation_metrics_summary.csv =====
# Aggregate per variant: mean and std for each numeric metric

# Use per_seed_df (from heldout_metrics.json) as primary source
if not per_seed_df.empty:
    metric_cols = [c for c in per_seed_df.columns
                   if c not in ('variant', 'label', 'seed', 'gpu_id', 'best_epoch',
                                'checkpoint', '_source_file', 'eval_n', 'n_meta_bins',
                                'true_sparsity', 'asw_raw', 'best_val_loss')
                   and pd.api.types.is_numeric_dtype(per_seed_df[c])]

    agg_rows = []
    for variant_name, grp in per_seed_df.groupby('variant', observed=True):
        label = VARIANT_LABELS.get(variant_name, variant_name)
        row = {'variant': variant_name, 'label': label, 'n_seeds': len(grp)}
        for col in metric_cols:
            vals = grp[col].dropna()
            if len(vals) > 0:
                row[f'{col}_mean'] = float(vals.mean())
                row[f'{col}_std'] = float(vals.std(ddof=1)) if len(vals) > 1 else float('nan')
            else:
                row[f'{col}_mean'] = float('nan')
                row[f'{col}_std'] = float('nan')
        agg_rows.append(row)

    ablation_summary = pd.DataFrame(agg_rows)
    print(f'Built ablation summary: {len(ablation_summary)} variants x {len(ablation_summary.columns)} cols')
    display(ablation_summary)
else:
    ablation_summary = pd.DataFrame()
    print('[WARN] No per-seed data available; ablation summary is empty.')

In [ ]:
# Save ablation_metrics_summary.csv
if not ablation_summary.empty:
    ablation_summary.to_csv(OUTDIR / 'ablation_metrics_summary.csv', index=False)
    print('Saved:', OUTDIR / 'ablation_metrics_summary.csv')
else:
    pd.DataFrame({'note': ['no data available']}).to_csv(OUTDIR / 'ablation_metrics_summary.csv', index=False)
    print('[WARN] Empty ablation_metrics_summary.csv saved as placeholder')

In [ ]:
# ===== Build ablation_rankings.csv =====
# Rank variants by each primary metric, respecting direction (+1 higher better, -1 lower better)

if not ablation_summary.empty:
    ranking_rows = []
    for metric, direction in METRIC_DIRECTION.items():
        mean_col = f'{metric}_mean'
        if mean_col not in ablation_summary.columns:
            continue
        # Sort by mean: ascending if lower-is-better, descending if higher-is-better
        ascending = (direction == -1)
        ranked = ablation_summary[['variant', 'label', mean_col]].dropna().sort_values(
            mean_col, ascending=ascending
        ).reset_index(drop=True)
        for rank_idx, (_, r) in enumerate(ranked.iterrows()):
            ranking_rows.append({
                'metric': metric,
                'direction': 'lower_better' if direction == -1 else 'higher_better',
                'rank': rank_idx + 1,
                'variant': r['variant'],
                'label': r['label'],
                f'{metric}_mean': r[mean_col],
            })

    rankings_df = pd.DataFrame(ranking_rows)
    print(f'Built rankings: {len(rankings_df)} rows ({rankings_df["metric"].nunique()} metrics)')

    # Pivot to show rank per variant per metric
    if not rankings_df.empty:
        rank_pivot = rankings_df.pivot_table(
            index=['variant', 'label'], columns='metric', values='rank', observed=False
        ).reset_index()
        rank_pivot['avg_rank'] = rank_pivot.iloc[:, 2:].mean(axis=1)
        rank_pivot = rank_pivot.sort_values('avg_rank')
        print('\nRank pivot (avg rank across metrics):')
        display(rank_pivot)
else:
    rankings_df = pd.DataFrame()
    rank_pivot = pd.DataFrame()
    print('[WARN] Cannot build rankings: empty ablation summary.')

In [ ]:
# Save ablation_rankings.csv
if not rankings_df.empty:
    rankings_df.to_csv(OUTDIR / 'ablation_rankings.csv', index=False)
    print('Saved:', OUTDIR / 'ablation_rankings.csv')
else:
    pd.DataFrame({'note': ['no data available']}).to_csv(OUTDIR / 'ablation_rankings.csv', index=False)
    print('[WARN] Empty ablation_rankings.csv saved as placeholder')

## Join with Benchmarking Results (from notebook 05)

In [ ]:
# ===== Load benchmarking results =====
bench_table_path = BENCHMARK_DIR / 'benchmark_evaluation_table.csv'
bench_summary_path = BENCHMARK_DIR / 'benchmark_summary.json'

bench_available = False
if bench_table_path.exists():
    try:
        bench_table = pd.read_csv(bench_table_path)
        print(f'Loaded benchmark_evaluation_table.csv: {len(bench_table)} rows')
        print(f'  conditions: {bench_table["condition"].unique().tolist()}')
        bench_available = True
    except Exception as e:
        print(f'[WARN] Failed to read benchmark table: {e}')
        bench_table = pd.DataFrame()
else:
    print(f'[INFO] {bench_table_path} not found. Run 05_benchmarking_alignment.ipynb first to enable join.')
    bench_table = pd.DataFrame()

if bench_summary_path.exists():
    try:
        with open(bench_summary_path) as f:
            bench_summary = json.load(f)
        print(f'\nLoaded benchmark_summary.json:')
        for k, v in bench_summary.items():
            print(f'  {k}: {v}')
    except Exception as e:
        print(f'[WARN] Failed to read benchmark_summary.json: {e}')
        bench_summary = {}
else:
    bench_summary = {}

In [ ]:
# ===== Build ablation_vs_benchmark_join.csv =====
# Strategy:
#   - "full_model" exists in both ablation and benchmarking → direct join
#   - For ablation variants without benchmarking counterparts, list them with available metrics
#   - Join on shared condition/variant key

join_rows = []

if not ablation_summary.empty:
    for _, arow in ablation_summary.iterrows():
        variant = arow['variant']
        label = arow['label']
        row = {
            'variant': variant,
            'label': label,
            'source': 'ablation',
            'n_seeds': arow.get('n_seeds', float('nan')),
        }
        # Add ablation metrics
        for col in ablation_summary.columns:
            if col not in ('variant', 'label'):
                row[f'ablation_{col}'] = arow[col]

        # If benchmarking data available, try to match
        if bench_available and not bench_table.empty:
            # Try direct condition match
            bench_match = bench_table[bench_table['condition'] == variant]
            if not bench_match.empty:
                bm = bench_match.iloc[0]
                row['benchmark_condition'] = variant
                row['benchmark_match'] = 'direct'
                for bcol in bench_table.columns:
                    if bcol not in ('condition', 'model_seed', 'shuffle_seed'):
                        row[f'benchmark_{bcol}'] = bm[bcol] if pd.notna(bm[bcol]) else float('nan')
            else:
                # Fallback: match "full_model" benchmark for all variants as reference
                full_bench = bench_table[bench_table['condition'] == 'full_model']
                if not full_bench.empty:
                    row['benchmark_condition'] = 'full_model (reference only)'
                    row['benchmark_match'] = 'reference'
                    bm = full_bench.iloc[0]
                    for bcol in bench_table.columns:
                        if bcol not in ('condition', 'model_seed', 'shuffle_seed'):
                            row[f'benchmark_{bcol}'] = bm[bcol] if pd.notna(bm[bcol]) else float('nan')
                else:
                    row['benchmark_match'] = 'no_match'
        else:
            row['benchmark_match'] = 'benchmark_data_unavailable'

        join_rows.append(row)

join_df = pd.DataFrame(join_rows)
print(f'Built join table: {len(join_df)} rows')
display(join_df)

In [ ]:
# Save ablation_vs_benchmark_join.csv
if not join_df.empty:
    join_df.to_csv(OUTDIR / 'ablation_vs_benchmark_join.csv', index=False)
    print('Saved:', OUTDIR / 'ablation_vs_benchmark_join.csv')
else:
    pd.DataFrame({'note': ['no data available']}).to_csv(OUTDIR / 'ablation_vs_benchmark_join.csv', index=False)
    print('[WARN] Empty ablation_vs_benchmark_join.csv saved as placeholder')

## Visualizations

In [ ]:
# ===== Figure 1: Variant metric comparison (bar chart) =====
if HAS_MPL and not ablation_summary.empty:
    # Pick key metrics to plot
    plot_metrics = [
        ('heldout_mse', 'Heldout MSE', -1),
        ('global_corr_nonzero', 'Global Corr (nonzero)', +1),
        ('meta_cell_corr_by_time_bin', 'Cell Corr by Time Bin', +1),
        ('sparsity_gap', 'Sparsity Gap', -1),
        ('asw_cell_type_separation', 'ASW Cell-Type Separation', +1),
    ]
    available_metrics = [(m, label, d) for m, label, d in plot_metrics
                         if f'{m}_mean' in ablation_summary.columns]

    if available_metrics:
        n_plots = len(available_metrics)
        fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4), squeeze=False)

        variants = ablation_summary['variant'].tolist()
        labels = ablation_summary['label'].tolist()
        x = np.arange(len(variants))
        colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']

        for ax_idx, (metric, mlabel, direction) in enumerate(available_metrics):
            ax = axes[0, ax_idx]
            mean_col = f'{metric}_mean'
            std_col = f'{metric}_std'
            means = ablation_summary[mean_col].values
            stds = ablation_summary[std_col].values if std_col in ablation_summary.columns else np.zeros(len(means))

            bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors[:len(variants)], alpha=0.85)
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
            ax.set_title(mlabel, fontsize=10)
            ax.set_ylabel(mlabel)
            direction_label = '(lower better)' if direction == -1 else '(higher better)'
            ax.text(0.5, -0.18, direction_label, transform=ax.transAxes, ha='center', fontsize=7, color='grey')

        fig.suptitle('Ablation Variant Metrics (mean +/- std across seeds)', fontsize=12, y=1.02)
        plt.tight_layout()
        fig.savefig(FIGDIR / 'variant_metric_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved:', FIGDIR / 'variant_metric_comparison.png')
    else:
        print('[WARN] No plottable metrics found in ablation_summary')
else:
    print('Skipping figure 1: ' + ('matplotlib not available' if not HAS_MPL else 'no data'))

In [ ]:
# ===== Figure 2: Variant stability (error bars / box plot) =====
if HAS_MPL and not per_seed_df.empty:
    # Use per-seed data to show distribution across seeds
    primary_metric = 'heldout_mse'
    corr_metric = 'global_corr_nonzero'

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Panel A: heldout_mse per seed
    ax = axes[0]
    variant_order = sorted(per_seed_df['variant'].unique())
    data_groups = [per_seed_df[per_seed_df['variant'] == v][primary_metric].dropna().values
                   for v in variant_order]
    bp = ax.boxplot(data_groups, labels=[VARIANT_LABELS.get(v, v) for v in variant_order],
                    patch_artist=True)
    for patch, color in zip(bp['boxes'], colors[:len(variant_order)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(f'{primary_metric} (lower better)', fontsize=10)
    ax.tick_params(axis='x', rotation=30, labelsize=8)

    # Panel B: global_corr_nonzero per seed
    ax = axes[1]
    data_groups2 = [per_seed_df[per_seed_df['variant'] == v][corr_metric].dropna().values
                    for v in variant_order]
    bp2 = ax.boxplot(data_groups2, labels=[VARIANT_LABELS.get(v, v) for v in variant_order],
                     patch_artist=True)
    for patch, color in zip(bp2['boxes'], colors[:len(variant_order)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(f'{corr_metric} (higher better)', fontsize=10)
    ax.tick_params(axis='x', rotation=30, labelsize=8)

    fig.suptitle('Variant Stability Across Seeds', fontsize=12)
    plt.tight_layout()
    fig.savefig(FIGDIR / 'variant_stability_boxplot.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved:', FIGDIR / 'variant_stability_boxplot.png')
else:
    print('Skipping figure 2: ' + ('matplotlib not available' if not HAS_MPL else 'no per-seed data'))

## Results Summary

In [ ]:
# ===== Results summary cell =====
print('=' * 60)
print('ABLATION ANALYSIS SUMMARY')
print('=' * 60)

# 1) Best variant by primary metric
if not ablation_summary.empty:
    primary = 'heldout_mse'
    mean_col = f'{primary}_mean'
    if mean_col in ablation_summary.columns:
        best_row = ablation_summary.loc[ablation_summary[mean_col].idxmin()]
        print(f'\nBest variant by {primary} (lower better):')
        print(f'  {best_row["variant"]} ({best_row["label"]}): {best_row[mean_col]:.6f}')

        # Relative changes vs full_model
        full_row = ablation_summary[ablation_summary['variant'] == 'full_model']
        if not full_row.empty:
            full_val = full_row[mean_col].values[0]
            print(f'\nRelative to full_model ({primary}):')
            for _, row in ablation_summary.iterrows():
                if row['variant'] != 'full_model':
                    pct = 100.0 * (row[mean_col] - full_val) / (abs(full_val) + 1e-12)
                    direction = 'worse' if pct > 0 else 'better'
                    print(f'  {row["variant"]}: {row[mean_col]:.6f} ({pct:+.1f}%, {direction})')

    # Also show best by global_corr
    corr_col = 'global_corr_nonzero_mean'
    if corr_col in ablation_summary.columns:
        best_corr_row = ablation_summary.loc[ablation_summary[corr_col].idxmax()]
        print(f'\nBest variant by global_corr_nonzero (higher better):')
        print(f'  {best_corr_row["variant"]} ({best_corr_row["label"]}): {best_corr_row[corr_col]:.6f}')

# 2) Benchmark comparison
if bench_available:
    print(f'\nBenchmark comparison available:')
    print(f'  Conditions: {bench_table["condition"].unique().tolist()}')
    if 'full_model' in bench_table['condition'].values:
        fm_bench = bench_table[bench_table['condition'] == 'full_model']
        print(f'  Full model benchmark cosine: {fm_bench["transition_direction_cosine_mean"].values[0]:.4f}')
        print(f'  Full model benchmark MSE: {fm_bench["raw_mse"].values[0]:.4f}')
else:
    print(f'\n[INFO] Benchmark data not found. Run 05_benchmarking_alignment.ipynb to enable join.')

# 3) Data completeness report
print(f'\nData completeness:')
print(f'  heldout_metrics.json files found: {len(per_seed_df)}')
print(f'  Missing/unreadable entries: {len(missing_files)}')
if not per_seed_df.empty:
    print(f'  Variants: {per_seed_df["variant"].nunique()}')
    print(f'  Seeds per variant:')
    for v in VARIANT_NAMES:
        cnt = (per_seed_df['variant'] == v).sum()
        status = 'OK' if cnt >= 2 else ('PARTIAL' if cnt == 1 else 'MISSING')
        print(f'    {v}: {cnt} seeds [{status}]')

# 4) Output manifest
print(f'\nOutput files written to {OUTDIR}:')
for f in sorted(OUTDIR.glob('*.csv')):
    print(f'  {f.name}')
for f in sorted(FIGDIR.glob('*.png')):
    print(f'  figures/{f.name}')

print('\n' + '=' * 60)
print('ABLATION ANALYSIS COMPLETE')
print('=' * 60)